Dependencies

In [1]:
import torch
import torch.nn as nn

from torchvision.datasets import MNIST
from torchvision import transforms

from torchsummary import summary

import matplotlib.pyplot as plt

Load Data

In [2]:
train = MNIST('data', train=True, transform=transforms.ToTensor(), download=True)
test = MNIST('data', train=False,transform=transforms.ToTensor())

Batch Data?

In [3]:
train_loader = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test,batch_size=128)
### DataLoader() can also sample and run multithread over a set number of workers

Model Architecture

In [5]:
n_input = 784
n_dense_1 = 64
n_dense_2 = 64
n_dense_3 = 64
n_out = 10

In [6]:
model = nn.Sequential(
    nn.Linear(n_input, n_dense_1),
    nn.ReLU(),
    nn.Linear(n_dense_1, n_dense_2),
    nn.ReLU(),
    nn.Linear(n_dense_2, n_dense_3),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(n_dense_3,n_out)
    # the final softmax activation function will be defined in the model hyperparameter configuration
)

In [7]:
summary(model,(1, n_input))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                [-1, 1, 64]          50,240
              ReLU-2                [-1, 1, 64]               0
            Linear-3                [-1, 1, 64]           4,160
              ReLU-4                [-1, 1, 64]               0
            Linear-5                [-1, 1, 64]           4,160
              ReLU-6                [-1, 1, 64]               0
           Dropout-7                [-1, 1, 64]               0
            Linear-8                [-1, 1, 10]             650
Total params: 59,210
Trainable params: 59,210
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.23
Estimated Total Size (MB): 0.23
----------------------------------------------------------------


Hyperparameters Configuration

In [8]:
cost_fxn = nn.CrossEntropyLoss() # includes softmax activation

In [9]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

Train

In [10]:
# PyTorch doesn't have the metrics, we need to define our own accuracy calculation
def accuracy_pct(pred_y, true_y):
    _,prediction = torch.max(pred_y,1)
    correct = (prediction==true_y).sum().item()
    return (correct/true_y.shape[0]) * 100.0

In [11]:
n_batches = len(train_loader)
n_batches

469

In [12]:
n_epochs = 5

print(f'Training for {n_epochs} epoches.\n')

for epoch in range(n_epochs):
    avg_cost = 0.0
    avg_accuracy = 0.0

    for i, (X,y) in enumerate(train_loader):
        # forward propagation:
        X_flat = X.view(X.shape[0], -1)
        y_hat = model(X_flat)
        cost = cost_fxn(y_hat,y)
        avg_cost += cost/n_batches

        # bachprop and optimization via GD
        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        # calculate accuracy
        accuracy = accuracy_pct(y_hat,y)
        avg_accuracy += accuracy/n_batches
        if (i+1) % 100 == 0:
            print(f'Step {i+1}')
    
    print(f'Epoch {epoch+1}/{n_epochs} complete: Cost: {avg_cost:.3f}, Accuracy: {avg_accuracy:.1f}% \n')

print('Training complete:')

Training for 5 epoches.

Step 100
Step 200
Step 300
Step 400
Epoch 1/5 complete: Cost: 0.381, Accuracy: 89.0% 

Step 100
Step 200
Step 300
Step 400
Epoch 2/5 complete: Cost: 0.202, Accuracy: 94.6% 

Step 100
Step 200
Step 300
Step 400
Epoch 3/5 complete: Cost: 0.172, Accuracy: 95.5% 

Step 100
Step 200
Step 300
Step 400
Epoch 4/5 complete: Cost: 0.165, Accuracy: 95.7% 

Step 100
Step 200
Step 300
Step 400
Epoch 5/5 complete: Cost: 0.148, Accuracy: 96.2% 

Training complete:


Test Model

In [13]:
n_test_batches = len(test_loader)
n_test_batches

79

In [14]:
model.eval() # This disables dropout and batch norm

with torch.no_grad(): # This disables autograd, reducing memory consumption
    avg_test_cost = 0.0
    avg_test_acc = 0.0

    for X, y in test_loader:

        X_flat = X.view(X.shape[0],-1)
        y_hat = model(X_flat)

        cost = cost_fxn(y_hat, y)
        avg_test_cost += cost/n_test_batches

        test_accuracy = accuracy_pct(y_hat,y)
        avg_test_acc += test_accuracy/n_test_batches
    
print(f'Test cost: {avg_test_cost:.3f}, Test accuracy: {avg_test_acc:.1f}%')

Test cost: 0.150, Test accuracy: 96.3%
